## Analyze GitHub Reactions

<a href="https://colab.research.google.com/github/anaclaraaraujo/github_reaction_analysis/blob/main/ analyze_github_reactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Descrição do Código

Este código realiza uma análise de sentimentos e interações em issues e pull requests de projetos no GitHub. Ele executa as seguintes etapas:  

### **1️⃣ Carregamento e Validação do CSV**  
- O código carrega um arquivo CSV contendo dados de issues e pull requests.  
- Ele verifica se todas as colunas obrigatórias estão presentes. Caso contrário, gera um erro informando as colunas ausentes.  

### **2️⃣ Configuração da API da OpenAI**  
- O usuário fornece sua chave de API da OpenAI para permitir a análise de sentimento.  

### **3️⃣ Análise de Sentimento das Issues e Pull Requests**  
- Cada issue/pull request passa por uma análise de sentimento, classificando-a como **Positiva, Negativa ou Neutra**.  
- A análise leva em conta indícios de frustração, apreciação, confusão e urgência.  
- A resposta é fornecida em JSON, garantindo um formato estruturado.  

### **4️⃣ Processamento de Reações**  
- O código contabiliza reações como 👍 (`plus_one`), ❤️ (`heart`), 🎉 (`hooray`), 😆 (`laugh`), entre outras.  
- Ele converte as colunas de datas para o formato `datetime` e calcula o **tempo de resolução** das issues (em horas).  

### **5️⃣ Análises Realizadas**  
A partir dos dados, o código realiza diversas análises:  
✅ **Influência do tom emocional no tempo de resolução** – Calcula a média do tempo de resolução para cada tipo de sentimento.  
✅ **Correlação entre número de reações e tempo de resolução** – Avalia se issues com mais reações são resolvidas mais rápido.  
✅ **Reações predominantes em cada sentimento** – Identifica quais reações são mais comuns para sentimentos positivos, negativos e neutros.  
✅ **Comparação de sentimentos entre issues de bugs e novas funcionalidades** – Verifica se issues de bugs possuem mais sentimentos negativos do que novas funcionalidades.  

### **6️⃣ Análise Dinâmica do Impacto das Reações**  
- Para cada tipo de tarefa (**bug** ou **enhancement**), o código identifica:  
  - A **reação predominante**.  
  - O **sentimento predominante** associado à tarefa.  
  - O **impacto da reação no engajamento**, gerando uma explicação baseada no contexto.  

### **7️⃣ Geração de Gráficos**  
O código gera visualizações para facilitar a interpretação dos dados:  
📊 **Gráfico de tempo de resolução por sentimento**  
📊 **Mapa de calor da correlação entre reações e tempo de resolução**  
📊 **Distribuição de reações por tipo de sentimento**  
📊 **Comparação de sentimentos entre bugs e novas funcionalidades**  
📊 **Distribuição das reações por linguagem de programação**  
📊 **Sentimentos predominantes por linguagem**  

Os gráficos são salvos no diretório especificado no Google Drive.  

### **8️⃣ Geração de Relatório**  
Por fim, o código salva um relatório (`reactions_analysis.txt`) contendo:  
- Resumo da classificação de sentimentos.  
- Contagem das reações.  
- Explicações detalhadas da análise de sentimento.  
- Análises adicionais e estatísticas sobre tempo de resolução, correlação entre reações e sentimentos.  
- Impacto das reações nas tarefas.  

O relatório é salvo no Google Drive para consulta posterior.  

# Instalação das Dependências

In [18]:
!pip install openai==0.28 pandas matplotlib seaborn

# Importação de Bibliotecas e Configuração do Google Drive

In [19]:
from google.colab import drive
import pandas as pd
import os
import openai
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import getpass

# Montar Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Carregamento e Validação do CSV

In [20]:
# Caminho do arquivo CSV
file_path = "/content/drive/My Drive/database/database.csv"

# Carregar CSV
df = pd.read_csv(file_path, delimiter=',', dtype=str)

# Verificar colunas obrigatórias
required_columns = ["repository", "url_repo", "url", "language", "stars", "id", "number",
                    "title", "body", "state", "comments", "created_at", "updated_at",
                    "closed_at", "labels", "plus_one", "plus_minus", "laugh", "hooray",
                    "confused", "heart", "rocket", "eyes"]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"O arquivo CSV está faltando as seguintes colunas: {', '.join(missing_columns)}")

print("Arquivo CSV carregado com sucesso!")

Arquivo CSV carregado com sucesso!


# Configuração da API OpenAI

In [32]:
# Definir chave da API
api_key = getpass.getpass("Digite sua OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key


Digite sua OpenAI API Key: ··········


# Função de Análise de Sentimento

In [22]:
# Função para análise de sentimento detalhada
def classify_sentiment_detailed(text, title, url):
    try:
        response = openai.ChatCompletion.create(
          model="gpt-3.5-turbo",
          messages=[
              {"role": "system", "content": """
              You are an expert in sentiment analysis for GitHub issues and pull requests.
              Your task is to classify the sentiment (Positive, Negative, Neutral) and explain the emotional cues.
              Identify signs of frustration, appreciation, excitement, confusion, or urgency.
              Respond ONLY with a valid JSON object in the following format:
              {"sentiment": "Positive/Negative/Neutral", "explanation": "Brief explanation."}
              Do NOT include any extra text, comments, or formatting outside of the JSON.
              """},
              {"role": "user", "content": f"Analyze sentiment: {text} Title: {title} URL: {url}"}
          ],
          temperature=0.7,
          max_tokens=2000,
          timeout=600
      )

        response_content = response.get('choices', [])
        if not response_content:
            return {"sentiment": "Error", "explanation": "No choices in the response."}

        response_content = response_content[0]['message']['content'].strip()
        print("Resposta da API:", response_content)

        try:
          response_content = response['choices'][0]['message']['content'].strip()
          parsed_response = json.loads(response_content)
          if "sentiment" in parsed_response and "explanation" in parsed_response:
              parsed_response['explanation'] = f"Title: {title} URL: {url} - {parsed_response['explanation']}"
              return parsed_response
          else:
              return {"sentiment": "Error", "explanation": f"Invalid response format: {response_content}"}
        except json.JSONDecodeError:
            return {"sentiment": "Error", "explanation": f"Failed to decode JSON: {response_content}"}

    except Exception as e:
        return {"sentiment": "Error", "explanation": f"API request failed: {str(e)}"}

# Função para contar reações

In [23]:
# Função para contar reações
def analyze_reactions(data):
    reactions_count = Counter()
    for pr_or_issue in data:
        for reaction in ['plus_one', 'plus_minus', 'laugh', 'hooray', 'confused', 'heart', 'rocket', 'eyes']:
            reactions_count[reaction] += int(pr_or_issue.get(reaction, 0) or 0)
    return dict(reactions_count)


# Cálculo do tempo de resolução

In [24]:
# Converter colunas de data para datetime
df['created_at'] = pd.to_datetime(df['created_at']).dt.tz_localize(None)
df['updated_at'] = pd.to_datetime(df['updated_at']).dt.tz_localize(None)
df['closed_at'] = pd.to_datetime(df['closed_at'], errors='coerce').dt.tz_localize(None)

# Calcular tempo de resolução (em horas)
df['resolution_time'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600

# Análise de Sentimentos

In [25]:
# Lista para armazenar sentimentos
sentiments = []

# Loop para analisar cada issue
for issue in df.to_dict(orient='records'):
    text = f"Title: {issue['title']}. body: {issue.get('body', '')}"
    sentiment_result = classify_sentiment_detailed(text, issue['title'], issue['url'])
    sentiments.append(sentiment_result)

# Adicionar sentimentos ao DataFrame
df['sentiment'] = [s.get('sentiment', 'Error') for s in sentiments]
df['sentiment_explanation'] = [s.get('explanation', 'No explanation') for s in sentiments]

# Análise de Reações

In [26]:
# Contagem de reações
reactions_count = analyze_reactions(df.to_dict(orient='records'))

# Cálculo das Métricas de Análise

In [27]:
# 1️⃣ O tom emocional influencia o tempo de resolução das issues?
resolution_by_sentiment = df.groupby('sentiment')['resolution_time'].mean()

# 2️⃣ Issues com mais reações são resolvidas mais rápido?
df['total_reactions'] = df[['plus_one', 'heart', 'hooray', 'laugh', 'plus_minus',
                            'confused', 'rocket', 'eyes']].sum(axis=1)
correlation_reactions_resolution = df[['total_reactions', 'resolution_time']].corr()

# 3️⃣ Quais reações predominam em cada tipo de sentimento?
reaction_columns = ['plus_one', 'heart', 'hooray', 'laugh', 'plus_minus', 'confused', 'rocket', 'eyes']

# Converter colunas de reações para numérico
df[reaction_columns] = df[reaction_columns].apply(pd.to_numeric, errors='coerce')

# Agrupar reações por sentimento
reactions_by_sentiment = df.groupby('sentiment')[reaction_columns].sum()

# 4️⃣ Issues de bug têm um tom mais negativo que novas funcionalidades?
df['is_bug'] = df['labels'].apply(lambda x: any('bug' in label.lower() for label in str(x).split(',')))
df['is_enhancement'] = df['labels'].apply(lambda x: any('enhancement' in label.lower() for label in str(x).split(',')))

bug_sentiment = df[df['is_bug']]['sentiment'].value_counts()
enhancement_sentiment = df[df['is_enhancement']]['sentiment'].value_counts()

# Função para analisar o impacto das reações

In [28]:
import pandas as pd

# Função para analisar o impacto das reações com base no sentimento e nas reações predominantes
def analyze_dynamic_reactions_impact(df):
    reaction_impact_data = []

    # Para cada tipo de tarefa (bug ou enhancement), analisamos as reações predominantes
    for task_type, label in [('Bug', 'is_bug'), ('Enhancement', 'is_enhancement')]:
        task_data = df[df[label]]

        # Contar as reações predominantes
        reactions_by_type = task_data[['plus_one', 'heart', 'hooray', 'laugh', 'plus_minus', 'confused', 'rocket', 'eyes']].sum()

        # Identificar reações predominantes
        dominant_reactions = reactions_by_type.idxmax()
        dominant_count = reactions_by_type.max()

        # Obter o sentimento predominante da tarefa (positive, negative, neutral)
        dominant_sentiment = task_data['sentiment'].mode()[0]

        # Construir descrição de impacto dinâmica com base na reação e no sentimento
        if dominant_sentiment == 'Positive':
            impact = f"A reação predominante '{dominant_reactions}' reflete um alto nível de aprovação, " \
                     f"indicando um engajamento positivo. A quantidade de {dominant_count} reações " \
                     "sugere que a colaboração está fluindo bem e que os colaboradores estão entusiasmados."
        elif dominant_sentiment == 'Negative':
            impact = f"A reação predominante '{dominant_reactions}' junto ao sentimento negativo indica que há " \
                     f"confusão ou desacordo. A quantidade de {dominant_count} reações sugere que há " \
                     "necessidade de esclarecimento ou revisão antes de avançar."
        else:  # Neutral sentiment
            impact = f"A reação predominante '{dominant_reactions}' e o sentimento neutro indicam um " \
                     "engajamento moderado, sem fortes emoções envolvidas. A quantidade de " \
                     f"{dominant_count} reações sugere uma colaboração tranquila, mas sem grande impulso."

        # Adicionar a linha à lista de impactos
        reaction_impact_data.append({
            'Tipo de Correção': task_type,
            'Tipo de Reação': dominant_reactions,
            'Impacto no Engajamento': impact,
            'Contagem de Reação': dominant_count,
            'Sentimento Predominante': dominant_sentiment
        })

    return pd.DataFrame(reaction_impact_data)

# Execução da análise de impacto das reações

In [29]:
# Executar a análise de impacto das reações dinâmica
reaction_impact_df = analyze_dynamic_reactions_impact(df)

# Exibir a tabela no Colab
from IPython.display import display
display(reaction_impact_df)


,Tipo de Correção,Tipo de Reação,Impacto no Engajamento,Contagem de Reação,Sentimento Predominante
0,Bug,plus_one,A reação predominante 'plus_one' e o sentiment...,668,Error
1,Enhancement,plus_one,A reação predominante 'plus_one' e o sentiment...,485,Error


# Geração de Gráficos

In [30]:
# Função para salvar gráficos
def save_plot(filename, tight=True):
    if tight:
        plt.tight_layout()
    plt.savefig(f"/content/drive/My Drive/analise/{filename}.png")
    plt.close()

# Gráfico 1: Influência do tom emocional no tempo de resolução
plt.figure(figsize=(8, 6))
resolution_by_sentiment.plot(kind='bar', color=['green', 'red', 'grey'])
plt.title('Tempo de Resolução Médio por Sentimento')
plt.xlabel('Sentimento')
plt.ylabel('Tempo de Resolução (Horas)')
plt.xticks(rotation=0)
save_plot("resolution_by_sentiment")

# Gráfico 2: Correlação entre reações e tempo de resolução
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_reactions_resolution, annot=True, cmap='coolwarm', center=0)
plt.title('Correlação entre Reações e Tempo de Resolução')
save_plot("correlation_reactions_resolution")

# Gráfico 3: Reações por Sentimento
plt.figure(figsize=(10, 6))
reactions_by_sentiment.plot(kind='bar', stacked=True)
plt.title('Reações por Tipo de Sentimento')
plt.xlabel('Sentimento')
plt.ylabel('Número de Reações')
plt.xticks(rotation=0)
save_plot("reactions_by_sentiment")

# Gráfico 4: Distribuição de Sentimentos de Bugs vs Enhancements
plt.figure(figsize=(8, 6))
bug_sentiment.plot(kind='bar', color='orange', label='Bug')
enhancement_sentiment.plot(kind='bar', color='lightblue', label='Enhancement')
plt.title('Distribuição de Sentimentos: Bug vs Enhancement')
plt.xlabel('Sentimento')
plt.ylabel('Número de Issues')
plt.legend()
plt.xticks(rotation=0)
save_plot("bug_vs_enhancement_sentiment")

# Gráfico 5: Agrupar por linguagem e somar as reações
reactions_by_language = df.groupby('language')[['plus_one', 'heart', 'hooray', 'laugh', 'plus_minus', 'confused', 'rocket', 'eyes']].sum()

fig, ax = plt.subplots(figsize=(16, 6))
reactions_by_language.plot(kind='bar', stacked=True, colormap='tab10', ax=ax)
plt.title('Distribuição das Reações por Linguagem', fontsize=16)
plt.xlabel('Linguagem', fontsize=14)
plt.ylabel('Quantidade de Reações', fontsize=14)
plt.xticks(rotation=30, ha='right', fontsize=12)
plt.legend(title="Reações", bbox_to_anchor=(1.2, 1), loc='upper left')
plt.subplots_adjust(left=0.05, right=0.75, top=0.9, bottom=0.3)
save_plot("reactions_by_language")

# Gráfico 6: Contagem de sentimentos por linguagem
sentiment_by_language = df.groupby('language')['sentiment'].value_counts().unstack().fillna(0)

fig, ax = plt.subplots(figsize=(16, 6))
sentiment_by_language.plot(kind='bar', stacked=True, colormap='coolwarm', ax=ax)
plt.title('Distribuição de Sentimentos por Linguagem', fontsize=16)
plt.xlabel('Linguagem', fontsize=14)
plt.ylabel('Quantidade de Sentimentos', fontsize=14)
plt.xticks(rotation=30, ha='right', fontsize=12)
plt.legend(title="Sentimento", bbox_to_anchor=(1.2, 1), loc='upper left')
plt.subplots_adjust(left=0.05, right=0.75, top=0.9, bottom=0.3)
save_plot("sentiment_by_language")

<Figure size 1000x600 with 0 Axes>

# Gerar o relatório

In [31]:
output_file = "/content/drive/My Drive/analise/reactions_analysis.txt"
# with open(output_file, "w") as f:
with open(output_file, "w", encoding="utf-8") as f:
    f.write("# Análise de Sentimento e Reações em Issues e Pull Requests\n\n")

    # Resumo da Classificação de Sentimento
    sentiment_summary = df['sentiment'].value_counts()
    f.write("## Resumo da Classificação de Sentimento:\n")
    f.write(sentiment_summary.to_string() + "\n\n")

    # Distribuição das Reações
    f.write("## Distribuição das Reações:\n")
    f.write(str(reactions_count) + "\n\n")

    # Explicações da Classificação de Sentimento
    f.write("## Explicações da Classificação de Sentimento:\n")
    for issue, sentiment in zip(df.to_dict(orient='records'), sentiments):
        f.write(f"- {sentiment['sentiment']}: {sentiment['explanation']}\n")

    # Análises adicionais
    f.write("\n## Análises Adicionais:\n")

    # 1. Influência do tom emocional no tempo de resolução:
    f.write("\n### Tempo de Resolução por Sentimento:\n")
    f.write(resolution_by_sentiment.to_string() + "\n")

    # 2. Correlação entre reações e tempo de resolução:
    f.write("\n### Correlação entre Reações e Tempo de Resolução:\n")
    f.write(str(correlation_reactions_resolution) + "\n")

    # 3. Reações por Sentimento:
    f.write("\n### Reações por Sentimento:\n")
    f.write(str(reactions_by_sentiment) + "\n")

    # 4. Distribuição de Sentimentos para Bugs vs Enhancements:
    f.write("\n### Distribuição de Sentimentos para Bugs vs Enhancements:\n")
    f.write(f"Bugs: {bug_sentiment.to_string()}\n")
    f.write(f"Enhancements: {enhancement_sentiment.to_string()}\n")

    # 5. Distribuição das Reações por Linguagem
    f.write("\n### Distribuição das Reações por Linguagem:\n")
    f.write(reactions_by_language.to_string() + "\n")

    # 6. Distribuição de Sentimentos por Linguagem
    f.write("\n### Distribuição de Sentimentos por Linguagem:\n")
    f.write(sentiment_by_language.to_string() + "\n")

    # 7. Adicionar a Tabela Dinâmica de Impacto das Reações:
    f.write("\n### Impacto das Reações nas Tarefas:\n")
    f.write(reaction_impact_df.to_string(index=False))

print(f"Relatório de Análise salvo em: {output_file}")

Relatório de Análise salvo em: /content/drive/My Drive/PFC/analise/reactions_analysis.txt
